# Aula 10 - Notebook: Avaliação Integrada do Módulo 1 — SCADA-Core Segurança & Diagnóstico

Este notebook integra todos os componentes desenvolvidos no **Módulo 1**: telemetria de sensores, discretização proposicional, prova de tautologias de segurança e motor especialista de inferência.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Any

class MapeadorProposicional:
    def extrair_proposicoes(self, telemetria: Dict[str, float]) -> Dict[str, bool]:
        return {
            'p1': telemetria.get('PT-101', 0.0) >= 180.0,
            't1': telemetria.get('TT-101', 0.0) >= 200.0,
            'g1': telemetria.get('AT-101', 0.0) >= 25.0,
            'e1': bool(telemetria.get('ESD-100', 0)),
            'v1': bool(telemetria.get('XV-101', 0)),
        }

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []
    def adicionar_regra(self, id_r, antecedentes, consequente, desc, prioridade=1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

class MotorInferencia:
    def __init__(self, base_conhecimento):
        self.bc = base_conhecimento
    def forward_chaining(self, fatos_iniciais):
        fatos_conhecidos = set(fatos_iniciais)
        historico = []
        passo = 1
        novos = True
        while novos:
            novos = False
            for regra in sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True):
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico.append({"Passo": passo, "Regra": regra.id_regra, "Diagnóstico": regra.descricao_diagnostico})
                    passo += 1
                    novos = True
                    break
        return fatos_conhecidos, historico

class SCADACoreModulo1:
    def __init__(self):
        self.mapeador = MapeadorProposicional()
        self.bc = BaseConhecimento()
        self.bc.adicionar_regra("R-01", ["p1", "t1"], "reacao_runaway", "Exotermia Descontrolada", 10)
        self.bc.adicionar_regra("R-02", ["reacao_runaway", "v1"], "trip_nh3", "Fechamento Imediato Válvula NH3", 10)
        self.motor = MotorInferencia(self.bc)
        
    def processar_ciclo_scan(self, telemetria: Dict[str, float]) -> Dict[str, Any]:
        props = self.mapeador.extrair_proposicoes(telemetria)
        fatos_ativos = {k for k, v in props.items() if v}
        trip = props['p1'] or props['t1'] or props['g1'] or props['e1']
        fatos_inf, trilha = self.motor.forward_chaining(fatos_ativos)
        return {
            "Trip_Ativo": trip,
            "Diagnósticos": list(fatos_inf)
        }

core1 = SCADACoreModulo1()
res1 = core1.processar_ciclo_scan({'PT-101': 195.0, 'TT-101': 210.0, 'XV-101': 1.0})
print("Resultado Scan Avaliação Módulo 1:", res1)
assert res1["Trip_Ativo"] is True
assert "trip_nh3" in res1["Diagnósticos"]
print("[OK] Avaliação Módulo 1 concluída com 100% de sucesso!")


Resultado Scan Avaliação Módulo 1: {'Trip_Ativo': True, 'Diagnósticos': ['v1', 't1', 'p1', 'trip_nh3', 'reacao_runaway']}
[OK] Avaliação Módulo 1 concluída com 100% de sucesso!
